# 4 2 content analysis he high mis

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections
source_corrections = get_source_corrections(window_days=28).query('prior_corrections==0').reset_index(drop=True)
source_corrections_matched = pd.read_parquet('data/source_corrections_content_matched.parquet')
source_corrections = source_corrections.merge(source_corrections_matched[['tweet_id']], on='tweet_id')
print(source_corrections.shape)
source_corrections.head()

In [ ]:
source_corrections = source_corrections.query(f'UserPostMisleadingness>={source_corrections["UserPostMisleadingness"].quantile(0.75)}').reset_index(drop=True)

print(len(source_corrections.user_id.unique()))
print(source_corrections["UserPostMisleadingness"].median().round(3))

In [ ]:
from utils import StataDiD, he_content_factors, prepare_content_count, get_coefs

means_yes, std_errs_yes, stats_yes, pvals_yes, clls_yes, chls_yes = [], [], [], [], [], []
means_no, std_errs_no, stats_no, pvals_no, clls_no, chls_no = [], [], [], [], [], []
means_ratio, std_errs_ratio, stats_ratio, pvals_ratio, clls_ratio, chls_ratio = [], [], [], [], [], []

for item in he_content_factors:
    fvar, fvar_name, user_var, filename, tcol, external_threshold = item[0], item[1], item[2], item[3], item[4], item[5]
    print('------------------------------')
    print(fvar_name)
    if fvar == 'misleadingness':
        filename = 'original_posts_wide_high_mis.parquet.gzip'
        tcol = 'weeks_from_display'

    post_counts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
    post_counts_by_var = prepare_content_count(filename, fvar, tcol, external_threshold)
    post_counts = post_counts.merge(post_counts_by_var, on=['tweet_id', tcol], how='left')
    post_counts[f'tweet_count_{fvar}_yes'] = post_counts[f'tweet_count_{fvar}_yes'].fillna(0)
    post_counts[f'tweet_count_{fvar}_no'] = post_counts[f'tweet_count_{fvar}_no'].fillna(0)
    post_counts[f'{fvar}_ratio'] = post_counts[f'tweet_count_{fvar}_yes']/post_counts['tweet_count']
    post_counts = post_counts.merge(source_corrections, on='tweet_id', how='inner')
    stata_did = StataDiD(post_counts, 'treated', 'tweet_id', tcol, -4, 4)

    print('[Group 1]', end=' ')
    stata_did.tw_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_yes', random_unit=False, cluster='account_cluster')
    mean_yes, std_err_yes, stat_yes, pval_yes, cll_yes, chl_yes = get_coefs(f'qui ppmlhdfe', 2, 3, stat_type='z', irr_or=1, unit_group=True, full_return=True)

    print('[Group 0]', end=' ')
    stata_did.tw_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_no', random_unit=False, cluster='account_cluster')
    mean_no, std_err_no, stat_no, pval_no, cll_no, chl_no = get_coefs(f'qui ppmlhdfe', 2, 3, stat_type='z', irr_or=1, unit_group=True, full_return=True)

    print('Ratio', end=' ')
    stata_did.tw_did(reg=f'qui xtreg', dev=f'{fvar}_ratio', random_unit=False, cluster='account_cluster', condition=f'if tweet_count>0 & ({fvar}_ratio>=0 & {fvar}_ratio<=1)')
    mean_ratio, std_err_ratio, stat_ratio, pval_ratio, cll_ratio, chl_ratio = get_coefs(f'qui xtreg', 2, 3, stat_type='t', unit_group=True, full_return=True)

    means_yes.extend(mean_yes)
    std_errs_yes.extend(std_err_yes)
    stats_yes.extend(stat_yes)
    pvals_yes.extend(pval_yes)
    clls_yes.extend(cll_yes)
    chls_yes.extend(chl_yes)

    means_no.extend(mean_no)
    std_errs_no.extend(std_err_no)
    stats_no.extend(stat_no)
    pvals_no.extend(pval_no)
    clls_no.extend(cll_no)
    chls_no.extend(chl_no)

    means_ratio.append(mean_ratio)
    std_errs_ratio.append(std_err_ratio)
    stats_ratio.append(stat_ratio)
    pvals_ratio.append(pval_ratio)
    clls_ratio.append(cll_ratio)
    chls_ratio.append(chl_ratio)

yes_label = 'Yes'
no_label = 'No'
ratio_label = 'Ratio'

hes_yes = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [yes_label]*len(he_content_factors),
                        'est': means_yes, 'std_err': std_errs_yes, 'stat': stats_yes, 'pval': pvals_yes,
                        'cll': clls_yes, 'chl': chls_yes})
hes_no = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [no_label]*len(he_content_factors),
                        'est': means_no, 'std_err': std_errs_no, 'stat': stats_no, 'pval': pvals_no,
                        'cll': clls_no, 'chl': chls_no})

hes_groups = pd.concat([hes_yes, hes_no], axis=0, ignore_index=True)

hes_ratio = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [ratio_label]*len(he_content_factors),
                        'est': means_ratio, 'std_err': std_errs_ratio, 'stat': stats_ratio, 'pval': pvals_ratio,
                        'cll': clls_ratio, 'chl': chls_ratio})

In [ ]:
import matplotlib.pyplot as plt
from utils import FONT_SIZE, vermilion, LW, MZ

def content_he_plot(hes_groups_df, hes_ratio_df, fig_size=(20, 8), cip=95, x_label='Estimated DiD effect', delta_x=-0.22, xlims=(-0.22, 0.462), output_name='he_content_sensitivity'):
    plt.style.use('default')
    plt.rcParams['font.size'] = FONT_SIZE
    split_colors = {'Yes': vermilion, 'No': vermilion}
    split_markers = {'Yes': 's', 'No': 'D'}
    fig = plt.figure(figsize=fig_size)
     # -------------------------
    # 1. Build y positions
    # -------------------------
    rows = []
    y = 0
    for group, gdf in hes_groups_df.groupby('group', sort=False):
        rows.append((group, '', None, None, None, None, None, y, True))  # group header
        y += 1
        for _, r in gdf.iterrows():
            rows.append((r.group, r.label, r.est, r.cll, r.chl, r.stat, r.pval, y, False))
            y += 1
        y += 0.5  # spacing between groups

    plot_df = pd.DataFrame(rows, columns=[
        "group", "label", "est", "ll", "hl", "stat", "pval", "y", "is_header"
    ])

    # -------------------------
    # 2. Figure layout
    # -------------------------
    ax_text = fig.add_axes([0.0, 0.0, 0.65, 1.0])  # left text
    ax_plot = fig.add_axes([0.68, 0.0, 0.3, 1.0])  # right forest

    # -------------------------
    # 3. Plot forest (right)
    # -------------------------
    ax_plot.axvline(0, linestyle="--", color="lightgray", linewidth=LW)
    for _, r in plot_df.iterrows():
        if not r.is_header:
            ax_plot.plot([r.ll, r.hl], [r.y, r.y], color="gray", linewidth=LW)
            ax_plot.plot(r.est, r.y, split_markers[r.label], color=split_colors[r.label], markersize=MZ)

    ax_plot.set_ylim(-1, plot_df["y"].max() + 1)
    ax_plot.invert_yaxis()
    ax_plot.set_xlabel(x_label, weight="bold")
    if xlims:
        ax_plot.set_xlim(xlims[0], xlims[1])

    # clean look
    ax_plot.spines["top"].set_visible(False)
    ax_plot.spines["right"].set_visible(False)
    ax_plot.spines["left"].set_visible(False)
    ax_plot.set_yticks([])

    # -------------------------
    # 5. Left text columns
    # -------------------------
    ax_text.set_xlim(0, 1)
    ax_text.set_ylim(-1, plot_df["y"].max() + 1)
    ax_text.invert_yaxis()
    ax_text.axis("off")

    for _, r in plot_df.iterrows():
        y = r.y

        if r.is_header:
            ax_text.text(0.0, y, r.group, weight="bold", va="center")
        else:
            # label
            ax_text.text(0.05, y, r.label, va="center")

            combined = f"${r.est:.3f}$, {cip}% CI$=[{r.ll:.3f}, {r.hl:.3f}]$, {r.stat}, {r.pval}"
            ax_text.text(0.25, y, combined, va="center")

    # -------------------------
    # 5. Optional row shading
    # -------------------------
    for i, r in plot_df.iterrows():
        if not r.is_header:
            ax_plot.axhspan(r.y - 0.5, r.y + 0.5, color="0.95", zorder=0)
        if r.is_header:
            ax_plot.text(x=delta_x, y=r.y,
                         s=f'Conditional share $={hes_ratio_df[hes_ratio_df.group == r.group]["est"].values[0][0]:.3f}$ {hes_ratio_df[hes_ratio_df.group == r.group]["pval"].values[0][0]}',
                         va="center", fontdict={"fontsize": FONT_SIZE - 2})
    # -------------------------
    plt.savefig(f'outputs/figures/{output_name}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
content_he_plot(hes_groups, hes_ratio, fig_size=(18, 8))

### Conditional misleadingness

In [ ]:
from pystata import stata
from utils import SEED

means_yes, std_errs_yes, stats_yes, pvals_yes, clls_yes, chls_yes = [], [], [], [], [], []
means_no, std_errs_no, stats_no, pvals_no, clls_no, chls_no = [], [], [], [], [], []
means_ratio, std_errs_ratio, stats_ratio, pvals_ratio, clls_ratio, chls_ratio = [], [], [], [], [], []

he_content_factors = [('misleadingness', 'Misleading posts', 'UserPostMisleadingness',
                       'original_posts_wide_high_mis.parquet.gzip', 'weeks_from_display', 0.86, 'sdid_posts_weekly_content_mis'),
                      ('misleadingness', 'Misleading posts (political or highly toxic)', 'UserPostMisleadingness',
                       'original_posts_wide_high_mis_politics.parquet.gzip', 'weeks_from_display', 0.86, 'sdid_posts_weekly_content_mis_politics'),
                      ('misleadingness', 'Misleading posts (non-political and low-toxicity)', 'UserPostMisleadingness',
                       'original_posts_wide_high_mis_non_politics.parquet.gzip', 'weeks_from_display', 0.86, 'sdid_posts_weekly_content_mis_non_politics')]
for item in he_content_factors:
    fvar, fvar_name, user_var, filename, tcol, external_threshold, figname = item[0], item[1], item[2], item[3], item[4], item[5], item[6]
    print('------------------------------')
    print(fvar_name)
    post_counts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
    post_counts_by_var = prepare_content_count(filename, fvar, tcol, external_threshold)
    post_counts = post_counts.merge(post_counts_by_var, on=['tweet_id', tcol], how='left')
    post_counts[f'tweet_count_{fvar}_yes'] = post_counts[f'tweet_count_{fvar}_yes'].fillna(0)
    post_counts[f'tweet_count_{fvar}_no'] = post_counts[f'tweet_count_{fvar}_no'].fillna(0)
    post_counts[f'{fvar}_ratio'] = post_counts[f'tweet_count_{fvar}_yes']/post_counts['tweet_count']
    post_counts = post_counts.merge(source_corrections, on='tweet_id', how='inner')
    stata_did = StataDiD(post_counts, 'treated', 'tweet_id', tcol, -4, 4)

    print('[Group 1]', end=' ')
    stata_did.tw_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_yes', random_unit=False, cluster='account_cluster')
    mean_yes, std_err_yes, stat_yes, pval_yes, cll_yes, chl_yes = get_coefs(f'qui ppmlhdfe', 2, 3, stat_type='z', irr_or=1, unit_group=True, full_return=True)

    stata.run(
        f"""
        sdid tweet_count_{fvar}_yes unitId weeks_from_display treated_after, ///
        vce(bootstrap) cluster(account_cluster) ///
        xline_opts(lcolor(none)) ///
        reps(100) seed({SEED}) graph ///
        g2_opt(xline(0, lcolor(black) lpattern(dash)) xtitle("Weeks from note display") ytitle("Post count") xlabel(-4(1)4) ylabel(0(5)25) title("") graphregion(color(white)))
        """
    )
    stata.run(f"graph export outputs/figures/{figname}.pdf, replace")

    print('[Group 0]', end=' ')
    stata_did.tw_did(reg=f'qui ppmlhdfe', dev=f'tweet_count_{fvar}_no', random_unit=False, cluster='account_cluster')
    mean_no, std_err_no, stat_no, pval_no, cll_no, chl_no = get_coefs(f'qui ppmlhdfe', 2, 3, stat_type='z', irr_or=1, unit_group=True, full_return=True)

    print('Ratio', end=' ')
    stata_did.tw_did(reg=f'qui xtreg', dev=f'{fvar}_ratio', random_unit=False, cluster='account_cluster', condition=f'if tweet_count>0 & ({fvar}_ratio>=0 & {fvar}_ratio<=1)')
    mean_ratio, std_err_ratio, stat_ratio, pval_ratio, cll_ratio, chl_ratio = get_coefs(f'qui xtreg', 2, 3, stat_type='t', unit_group=True, full_return=True)

    means_yes.extend(mean_yes)
    std_errs_yes.extend(std_err_yes)
    stats_yes.extend(stat_yes)
    pvals_yes.extend(pval_yes)
    clls_yes.extend(cll_yes)
    chls_yes.extend(chl_yes)

    means_no.extend(mean_no)
    std_errs_no.extend(std_err_no)
    stats_no.extend(stat_no)
    pvals_no.extend(pval_no)
    clls_no.extend(cll_no)
    chls_no.extend(chl_no)

    means_ratio.append(mean_ratio)
    std_errs_ratio.append(std_err_ratio)
    stats_ratio.append(stat_ratio)
    pvals_ratio.append(pval_ratio)
    clls_ratio.append(cll_ratio)
    chls_ratio.append(chl_ratio)

yes_label = 'Yes'
no_label = 'No'
ratio_label = 'Ratio'

hes_yes = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [yes_label]*len(he_content_factors),
                        'est': means_yes, 'std_err': std_errs_yes, 'stat': stats_yes, 'pval': pvals_yes,
                        'cll': clls_yes, 'chl': chls_yes})
hes_no = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [no_label]*len(he_content_factors),
                        'est': means_no, 'std_err': std_errs_no, 'stat': stats_no, 'pval': pvals_no,
                        'cll': clls_no, 'chl': chls_no})

hes_groups = pd.concat([hes_yes, hes_no], axis=0, ignore_index=True)

hes_ratio = pd.DataFrame({'group': [item[1] for item in he_content_factors],
                        'label': [ratio_label]*len(he_content_factors),
                        'est': means_ratio, 'std_err': std_errs_ratio, 'stat': stats_ratio, 'pval': pvals_ratio,
                        'cll': clls_ratio, 'chl': chls_ratio})

In [ ]:
content_he_plot(hes_groups, hes_ratio, output_name='he_content_sensitivity_mis', fig_size=(18, 5))

In [ ]:
from statsmodels.stats.multitest import multipletests

def adjust_pvalues(p_values):
    return {
        method: multipletests(p_values, method=method)[1]
        for method in ('holm', 'fdr_bh')
    }

adjusted = adjust_pvalues([0.028, 0.170, 0.308, 0.462, 0.006, 0.137])
print('Holm &   FDR-BH')
for holm, bh in zip(adjusted['holm'], adjusted['fdr_bh']):
    print(f'$p={holm:.3f}$  &   $p={bh:.3f}$')